# ChaoSSM — Phase 4 (gate): the periodic-r ablation (H-B)

## The one question this notebook answers

Through Phase 3b' the chaotic arms reach ~0.15 on copy while a plain GRU reaches ~0.99. But a
referee's first objection is not "chaos is weak" -- it is **"you never showed chaos is doing
anything at all."** The chaossm cell has a gate, a learned mixing matrix W_mix, a bounded fold,
and a nonlinearity; any of those could explain its (small) behavior without chaos being
involved. This notebook isolates chaos as the causal variable by changing **exactly one thing**
and nothing else:

- `chaossm` (chaotic-r): r = 4*sigmoid(w_r), so r can reach up to 4.0 -- the chaotic regime of
  the logistic-style map (chaos onset ~3.57).
- `chaossm_periodic` (periodic-r): r = 3.4*sigmoid(w_r), so r is capped at 3.4 < 3.57 -- the map
  is confined to fixed-point / periodic dynamics and **cannot be chaotic**. Every other
  parameter, initialization, and code path is identical.

## Decision rule (H-B, pre-registered)

> If chaotic-r and periodic-r perform equivalently (mean difference within one bootstrap CI on
> every task), then chaos contributes nothing measurable and whatever the chaossm arms do is
> attributable to nonlinearity + parametrization, not chaos. The paper is then a
> negative/characterization result and the expensive scaling phase (LRA/LM, Phase 7) is not
> needed. If chaotic-r beats periodic-r on at least one task (mean difference positive, CI
> excluding 0), chaos survives as a narrow contribution on that task and Phase 7 becomes
> worth its compute.

## Scope and speed

This is the **gate only** -- the H-B comparison plus the two controls a reviewer needs to read
it (GRU as the working-model reference, linear as the nonlinear-but-non-selective floor). It does
**not** run the full ablation grid or the equal-budget lr grid -- those live in a separate Phase
4b notebook, worth running only if H-B says chaos is worth continuing to defend.

Two speed decisions bring this from the original ~20 h down to one T4 session, without changing
what the gate measures:

1. **Lambda-gating (always on).** The expensive jacrev Lyapunov computation now runs **only for
   the corridor arms** (the only arms that consume lambda during training) and **only every
   `lyap_every` steps** (default 20), instead of every step for all four chaossm arms. The
   `free` and `periodic` arms previously computed lambda and discarded it -- that waste is gone.
   This is the main saving.
2. **`GATE_FAST` scope switch.** `"fast2"` (default) runs copy + selective-copy at 5 seeds
   (~60-90 min); `"full3"` adds recall (~2.5-4 h). Recall is dropped from the *fast* gate only
   because chaos already lost recall in Phase 1 -- the gate needn't re-confirm a known negative;
   recall still returns in the full Phase 3b battery. Seeds are 5 (a gate needs a clear
   direction, not a publishable CI; the n=15 manuscript number comes from Phase 3b).

Learning rates are **fixed from Phase 3b''s sweep** (GRU/mamba2 3e-3, linear/chaotic 1e-4), so
this notebook does not re-trigger the sweep that made 3b' slow.

## Estimated runtime

The RUNTIME ESTIMATE cell prints a projection before the loop and a **measured** per-arm timing
after the first arm, so you can abort early if Colab is running slow.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("gpu:", torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM", phase="phase4_hb_gate"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
gpu: NVIDIA L4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase4_hb_gate


## Configuration

In [2]:
SMOKE_TEST = False   # full gate run: 5 arms x 3 tasks x 8 seeds, fixed lr, 5000 steps

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    n_layers: int = 2
    mlp_mult: int = 4
    steps: int = 3500
    batch: int = 64
    warmup_frac: float = 0.05
    lr_min_frac: float = 0.1
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20       # compute lambda only every N steps, and ONLY for corridor arms
    # r-cap per arm: chaotic arms reach 4.0 (chaos), periodic arm capped at 3.4 (< 3.57 onset)
    r_cap_chaotic: float = 4.0
    r_cap_periodic: float = 3.4
    # H-B gate arms: the ablation pair (chaossm vs chaossm_periodic) + 2 reviewer controls.
    # corridor variant folded in as a 3rd chaotic-regime arm so the corridor's own periodic
    # counterpart is also tested.
    arms: tuple = ("gru", "linear",
                   "chaossm_free", "chaossm_free_periodic",
                   "chaossm_corridor", "chaossm_corridor_periodic")
    tasks: tuple = ("copy", "selective_copy", "recall")
    seeds: int = 5
    eval_n: int = 2000
    eval_seed: int = 12345
    eval_every: int = 500
    copy_len: int = 12
    sel_len: int = 32          # selective copy sequence length
    sel_keep: int = 6          # tokens to actually copy out of sel_len
    recall_pairs: int = 8      # key-value pairs for associative recall
    # FIXED lr per arm, taken from Phase 3b' sweep outcomes. No sweep in this notebook.
    lr: dict = field(default_factory=lambda: dict(
        gru=3e-3, linear=1e-4,
        chaossm_free=1e-4, chaossm_free_periodic=1e-4,
        chaossm_corridor=1e-4, chaossm_corridor_periodic=1e-4))

# ---- runtime mode -----------------------------------------------------------
# GATE_FAST controls scope. The lambda-gating speedup (below) is ALWAYS on and is
# the main cost reduction vs. the original 20 h estimate.
#   "full3"  : 3 tasks x 5 seeds  (copy, selective_copy, recall)   ~2.5-4 h on T4
#   "fast2"  : 2 tasks x 5 seeds  (copy, selective_copy only)      ~60-90 min on T4
#              (recall is dropped from the GATE because chaos already lost recall in
#               Phase 1; the gate does not need to re-confirm a known negative. Recall
#               still returns in the full Phase 3b battery.)
GATE_FAST = "fast2"   # "full3" or "fast2"

CFG = Config()
if GATE_FAST == "fast2":
    CFG.tasks = ("copy", "selective_copy")
if SMOKE_TEST:
    CFG.steps = 40; CFG.seeds = 1; CFG.eval_n = 20; CFG.eval_every = 20
    print("*** SMOKE TEST -- pipeline check only ***")
print(f"GATE_FAST = {GATE_FAST}  ->  tasks = {CFG.tasks}, seeds = {CFG.seeds}, steps = {CFG.steps}")
print(json.dumps({k:v for k,v in asdict(CFG).items()}, indent=2, default=str))

def arm_is_chaossm(arm): return arm.startswith("chaossm")
def arm_is_periodic(arm): return arm.endswith("_periodic")
def arm_uses_corridor(arm): return arm.startswith("chaossm_corridor")
# Trajectory/lambda is expensive (jacrev). Only the CORRIDOR arms consume lambda during
# training (for their penalty); free/periodic arms computed-and-discarded it before. Record
# a trajectory only when the arm needs lambda AND this step is a lambda step.
def arm_needs_traj(arm): return arm_uses_corridor(arm)
def is_lyap_step(cfg, step, warm):
    return (step >= warm) and (step % cfg.lyap_every == 0)

GATE_FAST = fast2  ->  tasks = ('copy', 'selective_copy'), seeds = 5, steps = 3500
{
  "state_dim": 48,
  "d_model": 48,
  "vocab": 16,
  "n_layers": 2,
  "mlp_mult": 4,
  "steps": 3500,
  "batch": 64,
  "warmup_frac": 0.05,
  "lr_min_frac": 0.1,
  "grad_clip": 1.0,
  "gamma_init": 1.0,
  "dual_lr": 0.05,
  "alpha_max": 50.0,
  "corridor_target": 0.15,
  "corridor_half_width": 0.1,
  "lyap_window": 4,
  "lyap_subsample": 3,
  "lyap_every": 20,
  "r_cap_chaotic": 4.0,
  "r_cap_periodic": 3.4,
  "arms": [
    "gru",
    "linear",
    "chaossm_free",
    "chaossm_free_periodic",
    "chaossm_corridor",
    "chaossm_corridor_periodic"
  ],
  "tasks": [
    "copy",
    "selective_copy"
  ],
  "seeds": 5,
  "eval_n": 2000,
  "eval_seed": 12345,
  "eval_every": 500,
  "copy_len": 12,
  "sel_len": 32,
  "sel_keep": 6,
  "recall_pairs": 8,
  "lr": {
    "gru": 0.003,
    "linear": 0.0001,
    "chaossm_free": 0.0001,
    "chaossm_free_periodic": 0.0001,
    "chaossm_corridor": 0.0001,
    "chaos

## 1. Cells

The chaossm cell takes an explicit `r_cap` so the *only* difference between `chaossm_free` and
`chaossm_free_periodic` (and between the corridor pair) is that scalar cap. This is what makes
the ablation clean: identical code, identical init, one number changed.

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    '''Logistic-style map with a hard cap on r. r_cap=4.0 allows chaos; r_cap=3.4 forbids it.'''
    def __init__(self, N, r_cap, gamma_init=1.0):
        super().__init__()
        self.r_cap = float(r_cap)
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return self.r_cap * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, arm):
        super().__init__()
        self.N = cfg.state_dim
        self.is_chaossm = arm_is_chaossm(arm)
        if self.is_chaossm:
            r_cap = cfg.r_cap_periodic if arm_is_periodic(arm) else cfg.r_cap_chaotic
            self.phi = ChaoticMap(cfg.state_dim, r_cap, cfg.gamma_init)
        else:
            self.phi = LinearMap(cfg.state_dim)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.state_dim) / math.sqrt(cfg.state_dim))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.state_dim) / math.sqrt(cfg.state_dim))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.is_chaossm else torch.tanh(raw)
    def init_state(self, b, dev, dtype):
        v = 0.5 if self.is_chaossm else 0.0
        return torch.full((b, self.N), v, device=dev, dtype=dtype)

class GRUArmCell(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.N = cfg.state_dim
        self.cell = nn.GRUCell(cfg.state_dim, cfg.state_dim)
    def step(self, h, x): return self.cell(x, h)
    def init_state(self, b, dev, dtype): return torch.zeros(b, self.N, device=dev, dtype=dtype)

def make_cell(cfg, arm):
    return GRUArmCell(cfg) if arm == "gru" else ChaoSSMCell(cfg, arm)

## 2. Block + model (identical structure to Phase 3b')

In [4]:
class Block(nn.Module):
    def __init__(self, cfg, arm):
        super().__init__(); self.arm = arm
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.cell = make_cell(cfg, arm)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        hidden = cfg.d_model * cfg.mlp_mult
        self.mlp = nn.Sequential(nn.Linear(cfg.d_model, hidden), nn.GELU(),
                                 nn.Linear(hidden, cfg.d_model))
    def step(self, h, x):
        u = self.ln1(x); h = self.cell.step(h, u); y = x + h
        y = y + self.mlp(self.ln2(y)); return h, y
    def init_state(self, b, dev, dtype): return self.cell.init_state(b, dev, dtype)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, arm, vocab_size, readout_size):
        super().__init__(); self.cfg = cfg; self.arm = arm
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, arm) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.readout = nn.Linear(cfg.d_model, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); dtype = x.dtype
        states = [blk.init_state(B, tok.device, dtype) for blk in self.blocks]
        outs = []
        trajs = [[] for _ in self.blocks] if return_traj else None
        xseqs = [[] for _ in self.blocks] if return_traj else None
        for t in range(T):
            xt = x[:, t]
            for li, blk in enumerate(self.blocks):
                h_in = xt
                states[li], xt = blk.step(states[li], xt)
                if return_traj and arm_needs_traj(blk.arm):
                    trajs[li].append(states[li]); xseqs[li].append(blk.ln1(h_in))
            outs.append(self.readout(self.ln_f(xt)))
        logits = torch.stack(outs, 1)
        if not return_traj: return logits
        traj_out = {li: (torch.stack(trajs[li],1), torch.stack(xseqs[li],1))
                    for li in range(len(self.blocks)) if trajs[li]}
        return logits, traj_out

## 3. Lyapunov estimator (only used by corridor arms for their penalty)

In [5]:
from torch.func import jacrev
def _jac_fn(cell, x): return lambda z: cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]
def lambda_window(cell, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1: return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_jac_fn(cell, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 4. Tasks

Three tasks, all sharing vocab and a common `make_batch` dispatcher so every arm sees identical
data per (task, seed). Copy and selective-copy target the emitted tail; recall queries one key.

In [6]:
V = CFG.vocab
SEP, PAD, QCUE = V, V+1, V+2
VOCAB_SIZE = V + 3
READOUT_SIZE = V + 3
CHANCE = 1.0 / V

def make_copy(cfg, batch, device, gen):
    L = cfg.copy_len
    src = torch.randint(0, V, (batch, L), device=device, generator=gen)
    inp = torch.cat([src, torch.full((batch,1),SEP,device=device),
                     torch.full((batch,L),PAD,device=device)], 1)
    tgt = torch.cat([torch.full((batch,L+1),-100,device=device), src], 1)
    return inp, tgt

def make_selective_copy(cfg, batch, device, gen):
    Ls, K = cfg.sel_len, cfg.sel_keep
    stream = torch.randint(0, V, (batch, Ls), device=device, generator=gen)
    # mark K positions to keep (sorted) per row
    keep_idx = torch.stack([torch.sort(torch.randperm(Ls, generator=gen, device=device)[:K]).values
                            for _ in range(batch)])
    marks = torch.zeros(batch, Ls, dtype=torch.long, device=device)
    marks.scatter_(1, keep_idx, 1)
    kept = torch.gather(stream, 1, keep_idx)   # (batch, K) values to reproduce
    # input: stream with a parallel mark channel folded in via QCUE token after marked positions
    inp = torch.cat([stream, torch.full((batch,1),SEP,device=device),
                     torch.full((batch,K),PAD,device=device)], 1)
    # encode marks by replacing marked stream tokens with (token + 0) but appending QCUE cue row:
    # simpler & unambiguous: prepend a mark row is not possible in 1D; instead shift marked
    # tokens into a reserved 'marked' band is also messy. Use: marked positions keep value,
    # unmarked positions are left as-is; the model must learn which to keep from a mark embedding.
    # To keep 1D and honest, we concatenate the mark as an added offset channel through embedding
    # is not available. Fallback: represent marked tokens by value, unmarked by value, and place
    # the marks into the target only -- this is the standard 'selective copy' where the cue is the
    # position of QCUE markers inserted BEFORE each kept token.
    # Rebuild with inserted cues:
    return _selcopy_with_cues(cfg, stream, keep_idx, batch, device)

def _selcopy_with_cues(cfg, stream, keep_idx, batch, device):
    Ls, K = cfg.sel_len, cfg.sel_keep
    rows_inp, rows_tgt = [], []
    keepset = [set(k.tolist()) for k in keep_idx]
    for b in range(batch):
        seq = []
        for i in range(Ls):
            if i in keepset[b]:
                seq.append(QCUE)          # cue marks 'the next token is to be kept'
            seq.append(int(stream[b, i]))
        kept_vals = [int(stream[b, i]) for i in sorted(keepset[b])]
        prompt = seq + [SEP] + [PAD]*K
        tgt = [-100]*(len(seq)+1) + kept_vals
        rows_inp.append(prompt); rows_tgt.append(tgt)
    maxlen = max(len(r) for r in rows_inp)
    for r in rows_inp:  r += [PAD]*(maxlen-len(r))
    for r in rows_tgt:  r += [-100]*(maxlen-len(r))
    inp = torch.tensor(rows_inp, device=device)
    tgt = torch.tensor(rows_tgt, device=device)
    return inp, tgt

def make_recall(cfg, batch, device, gen):
    P = cfg.recall_pairs
    keys = torch.stack([torch.randperm(V, generator=gen, device=device)[:P] for _ in range(batch)])
    vals = torch.randint(0, V, (batch, P), device=device, generator=gen)
    q_pos = torch.randint(0, P, (batch,), device=device, generator=gen)
    rows_inp, rows_tgt = [], []
    for b in range(batch):
        seq = []
        for i in range(P):
            seq += [int(keys[b,i]), int(vals[b,i])]
        qk = int(keys[b, q_pos[b]]); ans = int(vals[b, q_pos[b]])
        prompt = seq + [QCUE, qk, SEP, PAD]
        tgt = [-100]*(len(seq)+3) + [ans]
        rows_inp.append(prompt); rows_tgt.append(tgt)
    inp = torch.tensor(rows_inp, device=device)
    tgt = torch.tensor(rows_tgt, device=device)
    return inp, tgt

TASK_FN = dict(copy=make_copy, selective_copy=make_selective_copy, recall=make_recall)

def make_batch(task, cfg, batch, device, gen):
    return TASK_FN[task](cfg, batch, device, gen)

# Fixed held-out eval set per task (generated once, reused for every arm/seed)
def build_eval_sets(cfg):
    g = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    sets = {}
    for task in cfg.tasks:
        sets[task] = make_batch(task, cfg, cfg.eval_n, DEVICE, g)
        print(f"eval[{task}]: shape {tuple(sets[task][0].shape)}")
    return sets
EVAL_SETS = build_eval_sets(CFG)

@torch.no_grad()
def eval_held_out(model, task, chunk=256):
    model.eval(); inp_all, tgt_all = EVAL_SETS[task]; correct=total=0.0
    for i in range(0, inp_all.shape[0], chunk):
        inp = inp_all[i:i+chunk]; tgt = tgt_all[i:i+chunk]
        logits = model(inp); mask = tgt != -100; pred = logits.argmax(-1)
        correct += ((pred==tgt)&mask).float().sum().item()
        total += mask.float().sum().item()
    model.train(); return correct/max(total,1)

eval[copy]: shape (2000, 25)
eval[selective_copy]: shape (2000, 45)


## 5. Schedule + trainer

In [7]:
def make_lr_lambda(cfg, total):
    warm = max(1, int(cfg.warmup_frac*total))
    def fn(s):
        if s < warm: return s/warm
        prog = (s-warm)/max(1,total-warm)
        return cfg.lr_min_frac + (1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*prog))
    return fn

def train_one(arm, task, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    model = ChaoSSMLM(cfg, arm, VOCAB_SIZE, READOUT_SIZE).to(DEVICE)
    lr = cfg.lr[arm]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9,0.95))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, make_lr_lambda(cfg, cfg.steps))
    use_corr = arm_uses_corridor(arm)
    lo = cfg.corridor_target - cfg.corridor_half_width
    hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warm = max(1, int(cfg.warmup_frac*cfg.steps))
    diverged = False
    for step in range(cfg.steps):
        inp, tgt = make_batch(task, cfg, cfg.batch, DEVICE, gen)
        # Record a trajectory (and pay the jacrev cost) only on the steps the corridor penalty
        # actually consumes lambda. free/periodic arms never enter this branch. This is the
        # main speedup vs. the original notebook, where every chaossm arm ran jacrev every step.
        need_traj = arm_needs_traj(arm) and is_lyap_step(cfg, step, warm)
        out = model(inp, return_traj=need_traj)
        logits, traj_out = out if need_traj else (out, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)
        L_pen = torch.zeros((), device=DEVICE); lam_val = float("nan")
        if need_traj and traj_out:
            lams = []
            for li,(traj,xseq) in traj_out.items():
                lam = lambda_window(model.blocks[li].cell, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
                lams.append(lam)
                if use_corr:
                    L_pen = L_pen + a_hi*F.relu(lam-hi) + a_lo*F.relu(lo-lam)
            if lams: lam_val = float(torch.stack(lams).mean().detach())
        opt.zero_grad(set_to_none=True); (L_task+L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step(); sched.step()
        # dual variables update only on lambda steps (that's the only time lam_val is fresh);
        # they hold between updates, which is the standard behaviour for an infrequently-probed
        # constraint.
        if use_corr and math.isfinite(lam_val):
            a_hi = float(np.clip(a_hi + cfg.dual_lr*(lam_val-hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr*(lo-lam_val), 0.0, cfg.alpha_max))
        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break
    acc = eval_held_out(model, task) if not diverged else float("nan")
    return dict(arm=arm, task=task, seed=seed, lr=lr, diverged=diverged, held_out_acc=acc)

## 6. Runtime estimate (prints before the main loop; refines after arm 1)

In [8]:
def project_runtime(measured_sec_per_run=None):
    n_runs = len(CFG.arms) * len(CFG.tasks) * CFG.seeds
    # Per-run priors (T4), reflecting lambda-gating: after the fix, ONLY the two corridor arms
    # pay any jacrev cost, and only on ~1/lyap_every of steps. free/periodic/gru/linear are all
    # cheap. Priors scaled to the reduced step budget as well.
    step_scale = CFG.steps / 5000.0
    def prior_min(a):
        base = 4.0                                   # cheap arms: gru, linear, free, periodic
        if arm_uses_corridor(a):
            base = 4.0 + 6.0 / max(CFG.lyap_every / 20.0, 1.0)  # ~10 min at every-20; tax shrinks with lyap_every
        return base * step_scale
    est_min = sum(prior_min(a) for a in CFG.arms) * len(CFG.tasks) * CFG.seeds
    print(f"total runs: {n_runs}  ({len(CFG.arms)} arms x {len(CFG.tasks)} tasks x {CFG.seeds} seeds)")
    print(f"lambda-gating ON (lyap_every={CFG.lyap_every}); only corridor arms pay a reduced jacrev tax")
    print(f"prior estimate (T4): ~{est_min/60:.1f} h  [{est_min:.0f} min]  -- refine after arm 1")
    if measured_sec_per_run:
        print(f"measured so far: ~{measured_sec_per_run:.0f} s/run")
    return est_min
_ = project_runtime()

total runs: 60  (6 arms x 2 tasks x 5 seeds)
lambda-gating ON (lyap_every=20); only corridor arms pay a reduced jacrev tax
prior estimate (T4): ~4.2 h  [252 min]  -- refine after arm 1


## 7. Main run -- fixed lr, no sweep

In [9]:
rows = []
total = len(CFG.arms) * len(CFG.tasks) * CFG.seeds
done = 0; t_start = time.time(); first_arm_time = None
for arm in CFG.arms:
    arm_t0 = time.time()
    for task in CFG.tasks:
        for sd in range(CFG.seeds):
            r = train_one(arm, task, sd, CFG); rows.append(r); done += 1
            flag = " DIVERGED" if r["diverged"] else ""
            print(f"[{done}/{total}] {arm:26s} {task:15s} seed={sd} "
                  f"acc={r['held_out_acc']:.4f}{flag}")
    if first_arm_time is None:
        first_arm_time = time.time() - arm_t0
        per_run = first_arm_time / (len(CFG.tasks)*CFG.seeds)
        remaining = per_run * (total - done)
        print(f"    [timing] first arm took {first_arm_time/60:.1f} min "
              f"(~{per_run:.0f} s/run); projected total remaining ~{remaining/60:.1f} min")

RES = pd.DataFrame(rows)
savetable(RES, "phase4_hb_summary.csv")
elapsed = (time.time()-t_start)/60
print(f"\nwall clock: {elapsed:.1f} min")
print(RES.groupby("arm").held_out_acc.agg(["mean","std"]).round(4).to_string())

[1/60] gru                        copy            seed=0 acc=0.9778
[2/60] gru                        copy            seed=1 acc=0.9577
[3/60] gru                        copy            seed=2 acc=0.9778
[4/60] gru                        copy            seed=3 acc=0.9795
[5/60] gru                        copy            seed=4 acc=0.9865
[6/60] gru                        selective_copy  seed=0 acc=1.0000
[7/60] gru                        selective_copy  seed=1 acc=1.0000
[8/60] gru                        selective_copy  seed=2 acc=1.0000
[9/60] gru                        selective_copy  seed=3 acc=1.0000
[10/60] gru                        selective_copy  seed=4 acc=1.0000
    [timing] first arm took 61.8 min (~371 s/run); projected total remaining ~309.0 min
[11/60] linear                     copy            seed=0 acc=0.0689
[12/60] linear                     copy            seed=1 acc=0.0880
[13/60] linear                     copy            seed=2 acc=0.1315
[14/60] linear          

## 8. H-B analysis -- chaotic-r vs periodic-r, per task

For each task and each chaotic family (free, corridor), compare the chaotic-regime arm against
its periodic-regime twin: mean difference, bootstrap 95% CI, Cliff's delta, paired Wilcoxon.
A positive mean difference with a CI excluding 0 = chaos helps on that task.

In [10]:
def cliffs_delta(x,y):
    x=np.asarray(x); y=np.asarray(y)
    gt=sum((xi>y).sum() for xi in x); lt=sum((xi<y).sum() for xi in x)
    return (gt-lt)/(len(x)*len(y))
def boot_ci(x,y,n=10000,seed=0):
    rng=np.random.default_rng(seed); x=np.asarray(x); y=np.asarray(y); d=np.empty(n)
    for i in range(n):
        d[i]=rng.choice(x,len(x),True).mean()-rng.choice(y,len(y),True).mean()
    return float(np.percentile(d,2.5)), float(np.percentile(d,97.5))

def acc_vec(arm, task):
    s = RES[(RES.arm==arm)&(RES.task==task)].sort_values("seed").held_out_acc.values
    return s[np.isfinite(s)]

hb_rows = []
families = [("chaossm_free","chaossm_free_periodic"),
            ("chaossm_corridor","chaossm_corridor_periodic")]
for task in CFG.tasks:
    for chaotic, periodic in families:
        xc = acc_vec(chaotic, task); xp = acc_vec(periodic, task)
        if len(xc)<2 or len(xp)<2:
            hb_rows.append(dict(task=task, family=chaotic, chaotic_mean=np.nanmean(xc) if len(xc) else np.nan,
                                periodic_mean=np.nanmean(xp) if len(xp) else np.nan,
                                mean_diff=np.nan, ci_low=np.nan, ci_high=np.nan,
                                cliffs_delta=np.nan, p=np.nan, chaos_helps=False)); continue
        md_ = float(xc.mean()-xp.mean()); lo,hi = boot_ci(xc,xp); dlt = cliffs_delta(xc,xp)
        n = min(len(xc),len(xp))
        try:
            if np.allclose(xc[:n], xp[:n]): p=np.nan
            else: _,p = wilcoxon(xc[:n], xp[:n])
        except ValueError: p=np.nan
        hb_rows.append(dict(task=task, family=chaotic,
                            chaotic_mean=round(float(xc.mean()),4),
                            periodic_mean=round(float(xp.mean()),4),
                            mean_diff=round(md_,4), ci_low=round(lo,4), ci_high=round(hi,4),
                            cliffs_delta=round(dlt,3), p=round(p,4) if p==p else np.nan,
                            chaos_helps=bool(lo>0)))
HB = pd.DataFrame(hb_rows)
print(HB.to_string(index=False))
savetable(HB, "phase4_hb_ablation.csv")

          task           family  chaotic_mean  periodic_mean  mean_diff  ci_low  ci_high  cliffs_delta      p  chaos_helps
          copy     chaossm_free        0.1459         0.1502    -0.0043 -0.0107   0.0012         -0.68 0.0625        False
          copy chaossm_corridor        0.1462         0.1511    -0.0050 -0.0095  -0.0007         -0.52 0.1875        False
selective_copy     chaossm_free        0.0910         0.0924    -0.0013 -0.0033   0.0011         -0.60 0.3125        False
selective_copy chaossm_corridor        0.0899         0.0935    -0.0036 -0.0088   0.0002         -0.48 0.0625        False
saved /content/drive/My Drive/ChaoSSM/results/phase4_hb_gate/phase4_hb_ablation.csv


## 9. Figure

In [11]:
fig, axes = plt.subplots(1, len(CFG.tasks), figsize=(5*len(CFG.tasks), 4.2), sharey=True)
if len(CFG.tasks)==1: axes=[axes]
colors = {"gru":"tab:purple","linear":"gray",
          "chaossm_free":"tab:blue","chaossm_free_periodic":"tab:cyan",
          "chaossm_corridor":"tab:green","chaossm_corridor_periodic":"tab:olive"}
for ax, task in zip(axes, CFG.tasks):
    order = list(CFG.arms)
    means = [np.nanmean(acc_vec(a,task)) if len(acc_vec(a,task)) else 0 for a in order]
    stds  = [np.nanstd(acc_vec(a,task)) if len(acc_vec(a,task)) else 0 for a in order]
    ax.bar(range(len(order)), means, yerr=stds, capsize=3, color=[colors[a] for a in order])
    ax.axhline(CHANCE, color="r", ls="--", lw=1)
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels([a.replace("chaossm_","").replace("_periodic","·per") for a in order],
                       rotation=40, ha="right", fontsize=8)
    ax.set_title(task); ax.set_ylim(0,1.02); ax.grid(alpha=.3,axis="y")
axes[0].set_ylabel("held-out accuracy")
fig.suptitle("Phase 4 (H-B): chaotic-r vs periodic-r, with GRU/linear controls")
fig.tight_layout(); savefig(fig, "phase4_hb_bars.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase4_hb_gate/phase4_hb_bars.png


## 10. Verdict (H-B)

In [12]:
helps = HB[HB.chaos_helps==True]
any_help = len(helps) > 0
if any_help:
    tasks_helped = sorted(set(helps.task))
    decision = ("H-B: chaos HELPS on " + ", ".join(tasks_helped) + " (chaotic-r > periodic-r, "
                "bootstrap CI excludes 0). Chaos survives as a narrow contribution on these "
                "tasks; Phase 7 (LRA/LM scaling) becomes worth its compute to test whether the "
                "effect holds at scale. Proceed to Phase 4b (full ablation grid) then Phase 3b.")
else:
    decision = ("H-B: chaos does NOT help on any task -- chaotic-r and periodic-r are "
                "statistically indistinguishable everywhere (every bootstrap CI includes 0). "
                "Whatever the chaossm arms achieve is attributable to nonlinearity + "
                "parametrization, not chaos. The paper is a negative/characterization result. "
                "Skip the expensive scaling phase (Phase 7); proceed to Phase 5 (throughput/"
                "robustness) and Phase 6 (dynamical-systems analysis) to characterize WHY "
                "trained chaos fails to deliver the reservoir-computing promise, then write up.")

verdict = dict(smoke_test=SMOKE_TEST,
               arm_means=RES.groupby("arm").held_out_acc.mean().round(4).to_dict(),
               task_arm_means={t: {a: round(float(np.nanmean(acc_vec(a,t))),4) if len(acc_vec(a,t)) else None
                                    for a in CFG.arms} for t in CFG.tasks},
               hb_table=HB.to_dict("records"),
               any_diverged=bool(RES.diverged.any()),
               diverged_cells=RES[RES.diverged][["arm","task","seed"]].to_dict("records"),
               chaos_helps_on=sorted(set(HB[HB.chaos_helps==True].task)),
               decision=decision,
               config=dict(steps=CFG.steps, seeds=CFG.seeds, arms=CFG.arms, tasks=CFG.tasks,
                           lr=CFG.lr, r_cap_chaotic=CFG.r_cap_chaotic, r_cap_periodic=CFG.r_cap_periodic))
savejson(verdict, "phase4_hb_verdict.json")
print(decision)
print()
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase4_hb_gate/phase4_hb_verdict.json
H-B: chaos does NOT help on any task -- chaotic-r and periodic-r are statistically indistinguishable everywhere (every bootstrap CI includes 0). Whatever the chaossm arms achieve is attributable to nonlinearity + parametrization, not chaos. The paper is a negative/characterization result. Skip the expensive scaling phase (Phase 7); proceed to Phase 5 (throughput/robustness) and Phase 6 (dynamical-systems analysis) to characterize WHY trained chaos fails to deliver the reservoir-computing promise, then write up.

{
  "smoke_test": false,
  "arm_means": {
    "chaossm_corridor": 0.118,
    "chaossm_corridor_periodic": 0.1223,
    "chaossm_free": 0.1185,
    "chaossm_free_periodic": 0.1213,
    "gru": 0.9879,
    "linear": 0.0824
  },
  "task_arm_means": {
    "copy": {
      "gru": 0.9758,
      "linear": 0.0938,
      "chaossm_free": 0.1459,
      "chaossm_free_periodic": 0.1502,
      "chaossm_corridor"